In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
from shapely import wkt
import geopandas as gpd
from pathlib import Path
import matplotlib.cm as cm
import statsmodels.api as sm
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from scipy.stats import pearsonr
from shapely.geometry import Point
from sklearn.cluster import DBSCAN
from sklearn.cluster import KMeans
import matplotlib.patches as mpatches
from shapely.geometry import MultiPoint
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.outliers_influence import variance_inflation_factor

# Set global display format to show up to 6 decimal places
pd.options.display.float_format = '{:.6f}'.format

BASE_DIR = Path('/Users/wenlanzhang/Downloads/PhD_UCL/Data/Waste/')

# Load Waste Data

In [ ]:
df = pd.read_csv(BASE_DIR/"img/Combined_SVI.csv", low_memory=False) # all
# df = pd.read_csv(BASE_DIR/"img/Correct_SVI.csv") # only identified
# df = df[~df['img_dir'].isin(['Faith/', 'ZWL/'])]
# df[df['Domestic'] == 'Y']
# df

# Convert to a GeoDataFrame
df['geometry'] = df.apply(lambda row: Point(row['lon'], row['lat']), axis=1)
gdf_waste = gpd.GeoDataFrame(df, geometry='geometry', crs="EPSG:4326")
gdf_waste

# Load Social Economic data

In [ ]:
cols_to_drop = ['index', 'source_id', 'location', 'sublocation']

df_SE_2019 = pd.read_csv(BASE_DIR/"SE/cleaned_Nairobi_2019.csv")
# df_SE_2023 = pd.read_csv(BASE_DIR/"SE/cleaned_Nairobi_2023.csv")

# df_SE_2023 = df_SE_2023[df_SE_2023['County'] == 'Nairobi'] # If using Kenya
df_SE = df_SE_2019.drop (columns = cols_to_drop).reset_index() # 2019？

df_SE['geometry'] = df_SE['geometry'].apply(wkt.loads)
gdf_SE = gpd.GeoDataFrame(df_SE, geometry='geometry')
gdf_SE.set_crs(epsg=4326, inplace=True)
gdf_SE

# Match
some data is missing cause IDEAMaps has a larger boundary

In [ ]:
# Spatial join: assign each image to a boundary polygon
gdf_joined = gpd.sjoin(gdf_waste, gdf_SE, how="inner", predicate="within")
gdf_joined

# Summary of Metrics

| Metric                    | Purpose                                   |
| ------------------------- | ----------------------------------------- |
| `waste_pile_count`        | Raw count (biased by sampling effort)     |
| `waste_per_image`         | Adjusts for sampling effort               |
| `waste_density_per_km2`   | Adjusts for area size                     |
| `waste_per_image_per_km2` | Adjusts for both sampling effort and area |


In [ ]:
## Step 1: Metric 1 — Count of Waste Piles (Domestic == 'Y')

# Filter for actual waste detections
gdf_detected = gdf_joined[gdf_joined['Domestic'] == 'Y']

# Count domestic waste detections per admin unit
waste_counts = gdf_detected.groupby('index').size().reset_index(name='waste_pile_count')

## Step 2: Metric 2 — Count of All Images per Admin Unit (Sampling Effort)
# Count unique images per admin unit (sampling effort)
image_counts = gdf_joined.groupby('index')['img_name'].nunique().reset_index(name='image_count')
# image_counts

## Step 3: Metric 3 — Admin Area in km²
# Convert to projected CRS for area calculation
gdf_SE_proj = gdf_SE.to_crs(epsg=3395)
gdf_SE_proj['area_km2'] = gdf_SE_proj.geometry.area / 10**6

# Extract area for merging
area_df = gdf_SE_proj[['index', 'area_km2']]
# area_df

## Step 4: Merge All Metrics
# Merge counts
summary = pd.merge(image_counts, waste_counts, on='index', how='left')
summary = pd.merge(summary, area_df, on='index', how='left')

# Replace NaNs (e.g., if no waste found in a unit)
summary['waste_pile_count'] = summary['waste_pile_count'].fillna(0)
summary

In [ ]:
## Step 5: Compute Final Metrics
# Metric 2: Waste per image
summary['waste_per_image'] = summary.apply(
    lambda row: row['waste_pile_count'] / row['image_count'] if row['image_count'] > 0 else 0,
    axis=1
)

# Metric 3: Waste density per km²
summary['waste_density_per_km2'] = summary.apply(
    lambda row: row['waste_pile_count'] / row['area_km2'] if row['area_km2'] > 0 else 0,
    axis=1
)

# Metric 4: Waste per image per km²
summary['waste_per_image_per_km2'] = summary.apply(
    lambda row: row['waste_per_image'] / row['area_km2'] if row['area_km2'] > 0 else 0,
    axis=1
)
summary

In [ ]:
# Merge Back to GeoDataFrame for Mapping
gdf_SE_summary = gdf_SE.merge(summary, on='index', how='left')
gdf_SE_summary

In [ ]:
# gdf_SE_summary.to_csv(BASE_DIR/"Processed/Waste_by_admin5_withSE_2019.csv", index=True)

# Plot

In [ ]:
gdf_waste_correct = gdf_waste[gdf_waste['Domestic'] == 'Y']

In [ ]:
# import geopandas as gpd
# import matplotlib.pyplot as plt

# 1. Ensure both GeoDataFrames use the same coordinate reference system
gdf_SE_summary = gdf_SE_summary.to_crs(epsg=4326)
gdf_waste_correct = gdf_waste_correct.to_crs(epsg=4326)

# 2. Filter only 'existing' waste points
gdf_waste_existing = gdf_waste_correct[gdf_waste_correct['exist'] == True]

# 3. Spatial join: Keep only points within the administrative boundary
waste_within_admin = gpd.sjoin(
    gdf_waste_existing, 
    gdf_SE_summary[['geometry']],  # Keep only geometry to avoid duplicate columns
    how='inner', 
    predicate='within'
)

# 4. Plot
fig, ax = plt.subplots(figsize=(10, 10))

# Plot admin boundaries
gdf_SE_summary.boundary.plot(ax=ax, color='black', linewidth=1, label='Admin Boundaries')

# Plot filtered waste locations
waste_within_admin.plot(ax=ax, color='red', markersize=10, label='Waste Locations (within area)')

# Final touches
ax.set_title("Filtered Waste Locations Within Research Area")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.tight_layout()
plt.show()


# Exploring Social Economic Characters

## Clean up Data 

In [ ]:
# Step 1: Select numeric columns
df_numeric = gdf_SE_summary.select_dtypes(include=['float64', 'int64'])

# Step 2: Drop area or other non-informative columns
df_numeric = df_numeric.drop(columns=['area_km2'], errors='ignore')

# Step 3: Drop columns with all NaNs
df_numeric_clean = df_numeric.dropna(axis=1, how='all')

# Step 4: Drop constant columns (not useful for modeling)
df_numeric_clean = df_numeric_clean.loc[:, df_numeric_clean.nunique() > 1]

# Step 5: Drop columns that break correlation matrix (NaNs in corr matrix)
nan_corr_columns = df_numeric_clean.corr().isna().any()
to_drop = nan_corr_columns[nan_corr_columns].index.tolist()
df_numeric = df_numeric_clean.drop(columns=to_drop)
df_numeric

In [ ]:
# See how many NaNs and 0s in each column
nan_counts = df_numeric.isna().sum()
zero_counts = (df_numeric == 0).sum()

nan_zero_summary = pd.DataFrame({
    'NaNs': nan_counts,
    'Zeros': zero_counts,
    'Total': len(df_numeric),
    'NaN %': (nan_counts / len(df_numeric)) * 100,
    'Zero %': (zero_counts / len(df_numeric)) * 100
})

# Show only columns with at least some NaNs or 0s
nan_zero_summary = nan_zero_summary[(nan_zero_summary['NaNs'] > 0) | (nan_zero_summary['Zeros'] > 0)]
print(nan_zero_summary.sort_values(by='NaNs', ascending=False))

# df_numeric[df_numeric['waste_per_image'].isna()]

In [ ]:
# Fill NaNs with 0, and then remove columns with more than 5 0s

# Identify rows where image_count is 0
zero_image_rows = df_numeric['image_count'] == 0

# Replace NaNs with 0
df_numeric = df_numeric.fillna(0)

# Count number of zeros in each column
zero_counts = (df_numeric == 0).sum()

# Get column names with 5 or fewer zeros
columns_to_keep = zero_counts[zero_counts <= 5].index

# Keep only those columns
df_numeric = df_numeric[columns_to_keep]
df_numeric

## Overall - All features 

In [ ]:
df_numeric.describe().T  # T = transpose for better view

In [ ]:
# Correlation Matrix Heatmap (Broad View)

plt.figure(figsize=(12, 10))
corr_matrix = df_numeric.corr()
sns.heatmap(corr_matrix, cmap="coolwarm", annot=False, linewidths=0.5)
plt.title("Correlation Matrix of All Numerical Features")
plt.show()

## Select one feature 

In [ ]:
target = 'waste_per_image'
# target = 'waste_per_image_per_km2'

AnotherVariable = 'grdi'

### Correlation

In [ ]:
# Get correlation with all other features
correlations = df_numeric.corr()[target].sort_values(ascending=False)
print(correlations)

2023:  
- density_pop: R = 0.482, p-value = 9.94e-08  
- grdi: R = 0.419, p-value = 5.35e-06  
- nr_buildings: R = -0.340, p-value = 0.00028

2019:
- density_pop: R = 0.482, p-value = 9.94e-08  
- grdi: R = 0.418, p-value = 5.55e-06
- nr_buildings: R = -0.348, p-value = 0.000197

In [ ]:
# calculate R and p-value between target and one other variable
r_value, p_value = pearsonr(df_numeric[target], df_numeric[AnotherVariable])
print(f"R = {r_value:.6f}, p-value = {p_value:.6g}")

In [ ]:
correlations.drop(target).plot(kind='barh', figsize=(6, 16))
plt.title(f'Correlation of Features with {target}')
plt.xlabel('Correlation')
plt.grid(True)
plt.show()

In [ ]:
vif_data = pd.DataFrame()
X = df_numeric.drop(target, axis=1)
vif_data['feature'] = X.columns
vif_data['VIF'] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print(vif_data)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

x = df_numeric[AnotherVariable]
y = df_numeric[target]

sns.regplot(x=x, y=y, ci=95, line_kws={"color": "red"})
plt.title(f"Regression Line\nR = {r_value:.2f}, p = {p_value:.2e}")
plt.xlabel(AnotherVariable)
plt.ylabel(target)
plt.show()

In [ ]:
sns.boxplot(x=df_numeric[target])
plt.title("Boxplot of Target Variable")
plt.show()

In [ ]:
Q1 = df_numeric[target].quantile(0.25)
Q3 = df_numeric[target].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

df_clean = df_numeric[(df_numeric[target] >= lower_bound) & (df_numeric[target] <= upper_bound)]

In [ ]:
sns.regplot(x=AnotherVariable, y=target, data=df_numeric, ci=95, line_kws={"color": "red"})
plt.title("Linear Fit")
plt.show()

In [ ]:
import numpy as np
from numpy.polynomial.polynomial import Polynomial

# Fit 2nd degree
p2 = np.polyfit(x, y, deg=2)
x_range = np.linspace(x.min(), x.max(), 100)
y_pred2 = np.polyval(p2, x_range)

plt.scatter(x, y)
plt.plot(x_range, y_pred2, color='green', label='Quadratic Fit')
plt.legend()
plt.title("Polynomial Fit (Degree 2)")
plt.show()


In [ ]:
import statsmodels.api as sm

lowess = sm.nonparametric.lowess
lowess_result = lowess(y, x, frac=0.3)  # Adjust frac for smoothness

plt.scatter(x, y, alpha=0.5)
plt.plot(lowess_result[:, 0], lowess_result[:, 1], color='orange', label='LOWESS')
plt.title("LOWESS Fit")
plt.legend()
plt.show()

In [ ]:
from scipy.stats import pearsonr

x_clean = df_clean[AnotherVariable]
y_clean = df_clean[target]

r_clean, p_clean = pearsonr(x_clean, y_clean)
print(f"After outlier removal:\nR = {r_clean:.4f}, p = {p_clean:.4e}")


In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from sklearn.metrics import mean_squared_error, r2_score
from scipy.stats import pearsonr
import matplotlib.pyplot as plt

# Prepare variables
x = df_numeric[AnotherVariable]
y = df_numeric[target]

# Outlier Removal
Q1 = y.quantile(0.25)
Q3 = y.quantile(0.75)
IQR = Q3 - Q1
lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
df_clean = df_numeric[(y >= lower) & (y <= upper)]

x_clean = df_clean['grdi']
y_clean = df_clean[target]

# Container for results
results = []

def evaluate_model(name, x_vals, y_vals, y_pred):
    rmse = np.sqrt(mean_squared_error(y_vals, y_pred))
    r2 = r2_score(y_vals, y_pred)
    results.append({
        "Model": name,
        "Data": "Original" if x_vals.equals(x) else "Without Outliers",
        "R²": r2,
        "RMSE": rmse
    })

# 1. Linear Regression
coef = np.polyfit(x, y, 1)
y_pred_lin = np.polyval(coef, x)
evaluate_model("Linear", x, y, y_pred_lin)

coef_clean = np.polyfit(x_clean, y_clean, 1)
y_pred_lin_clean = np.polyval(coef_clean, x_clean)
evaluate_model("Linear", x_clean, y_clean, y_pred_lin_clean)

# 2. Polynomial Degree 2
coef2 = np.polyfit(x, y, 2)
y_pred_poly2 = np.polyval(coef2, x)
evaluate_model("Poly (deg 2)", x, y, y_pred_poly2)

coef2_clean = np.polyfit(x_clean, y_clean, 2)
y_pred_poly2_clean = np.polyval(coef2_clean, x_clean)
evaluate_model("Poly (deg 2)", x_clean, y_clean, y_pred_poly2_clean)

# 3. Polynomial Degree 3
coef3 = np.polyfit(x, y, 3)
y_pred_poly3 = np.polyval(coef3, x)
evaluate_model("Poly (deg 3)", x, y, y_pred_poly3)

coef3_clean = np.polyfit(x_clean, y_clean, 3)
y_pred_poly3_clean = np.polyval(coef3_clean, x_clean)
evaluate_model("Poly (deg 3)", x_clean, y_clean, y_pred_poly3_clean)

# 4. LOWESS
lowess = sm.nonparametric.lowess
y_pred_lowess = lowess(y, x, frac=0.3)[:, 1]
evaluate_model("LOWESS", x, y, y_pred_lowess)

y_pred_lowess_clean = lowess(y_clean, x_clean, frac=0.3)[:, 1]
evaluate_model("LOWESS", x_clean, y_clean, y_pred_lowess_clean)

# Convert to DataFrame
results_df = pd.DataFrame(results)
print(results_df)


| Model          | R² (Original) | R² (No Outliers) | RMSE (Original) | RMSE (No Outliers) |
| -------------- | ------------- | ---------------- | --------------- | ------------------ |
| **Linear**     | 0.17          | 0.02             | 0.0859          | 0.0161             |
| **Poly deg 2** | 0.26          | 0.03             | 0.0814          | 0.0160             |
| **Poly deg 3** | 0.27          | 0.06             | 0.0808          | 0.0157             |
| **LOWESS**     | -0.02         | -0.25            | 0.0956          | 0.0182             |

In [ ]:
sns.regplot(x='grdi', y=target, data=df_numeric)
plt.title("Linearity Check: grdi vs target")
plt.show()

In [ ]:
from sklearn.linear_model import LinearRegression

X = df_numeric[['grdi']]
y = df_numeric[target]

model = LinearRegression().fit(X, y)
residuals = y - model.predict(X)

plt.scatter(df_numeric['grdi'], residuals)
plt.axhline(0, color='red', linestyle='--')
plt.title("Residuals vs grdi (Check for heteroscedasticity)")
plt.show()


In [ ]:
import scipy.stats as stats

stats.probplot(residuals, dist="norm", plot=plt)
plt.title("Q-Q Plot of Residuals")
plt.show()


In [ ]:
from sklearn.metrics import r2_score, mean_squared_error
import numpy as np

y_pred = model.predict(X)
r2 = r2_score(y, y_pred)
rmse = np.sqrt(mean_squared_error(y, y_pred))

print(f"R²: {r2:.4f}")
print(f"RMSE: {rmse:.4f}")


In [ ]:
gdf_SE_summary

In [ ]:
import geopandas as gpd
import libpysal
from esda import Moran
import matplotlib.pyplot as plt
import seaborn as sns

# # Assuming your GeoDataFrame is named `df`
# # Step 1: Fix invalid geometries
# df["geometry"] = df["geometry"].buffer(0)

# Optional: drop empty or still-invalid geometries
gdf_SE_summary = gdf_SE_summary[~gdf_SE_summary["geometry"].is_empty]
gdf_SE_summary = gdf_SE_summary[gdf_SE_summary["geometry"].is_valid]

# Step 1: Prepare weights matrix (spatial relationships)
# Queen contiguity (shared borders)
w = libpysal.weights.Queen.from_dataframe(gdf_SE_summary)

# Row-standardize the weights
w.transform = 'r'

# Step 2: Compute Moran's I
moran = Moran(gdf_SE_summary['waste_per_image'], w)

# Step 3: Output results
print(f"Moran's I: {moran.I:.4f}")
print(f"p-value: {moran.p_sim:.4f}")


In [ ]:
# gdf_SE_summary['waste_per_image']
gdf_SE_summary["waste_per_image"].describe()
# gdf_SE_summary["waste_per_image"].isna().sum()


In [ ]:
import libpysal

# Step 1: Rebuild weights matrix
w = libpysal.weights.Queen.from_dataframe(gdf_SE_summary)
w.transform = 'r'

# Step 2: Check for disconnected areas
print("Isolated polygons with no neighbors:", w.islands)


In [ ]:
if w.islands:
    gdf_SE_summary = gdf_SE_summary.drop(index=w.islands)
    w = libpysal.weights.Queen.from_dataframe(gdf_SE_summary)
    w.transform = 'r'

    # Re-run Moran's I
    from esda import Moran
    moran = Moran(df["waste_per_image"], w)
    print(f"Moran's I: {moran.I:.4f}")
    print(f"p-value: {moran.p_sim:.4f}")


In [ ]:
gdf_SE_summary["waste_per_image"] = pd.to_numeric(gdf_SE_summary["waste_per_image"], errors='coerce')
print(gdf_SE_summary["waste_per_image"].isna().sum())

In [ ]:
from sklearn.preprocessing import StandardScaler

gdf_SE_summary["waste_z"] = StandardScaler().fit_transform(gdf_SE_summary[["waste_per_image"]])

from esda import Moran_Local

lisa = Moran_Local(gdf_SE_summary["waste_z"], w)
gdf_SE_summary["local_I"] = lisa.Is
gdf_SE_summary["p_sim"] = lisa.p_sim


In [ ]:
# Flag areas with statistically significant local autocorrelation
gdf_SE_summary["significant"] = lisa.p_sim < 0.05

# Print summary
print(gdf_SE_summary["significant"].value_counts())


### Map

In [ ]:
# Clean, rounded ticks manually (recommended)
ticks = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6]

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
gdf_SE_summary.plot(
    column=target,
    cmap='OrRd',
    linewidth=0.5,
    edgecolor='grey',
    legend=True,
    legend_kwds={
        'shrink': 0.6,
        'label': target.replace('_', ' ').title(),
        'orientation': 'vertical',
        'ticks': ticks
    },
    ax=ax
)

ax.set_title(f"{target.replace('_', ' ').title()} by Admin Unit", fontsize=15)
ax.set_axis_off()
plt.tight_layout()
plt.show()

In [ ]:
# Define tick values manually or using linspace
vmin = gdf_SE_summary[target].min()
vmax = gdf_SE_summary[target].max()
ticks = np.linspace(vmin, vmax, num=5)  # 5 ticks across the range

# Plot
fig, ax = plt.subplots(figsize=(10, 8))
gdf_SE_summary.plot(
    column=target,
    cmap='OrRd',
    scheme='Quantiles',   # or 'equal_interval', 'natural_breaks'
    k=4,                  # number of bins
    legend=True,
    edgecolor='white',    # White boundary for geometries
    linewidth=0.5,        # Thickness of the boundary
    ax=ax
)

ax.set_title(f"{target.replace('_', ' ').title()} by Admin Unit", fontsize=15)
ax.set_axis_off()
plt.tight_layout()
plt.show()

### Stats  

In [ ]:
X = df_numeric.drop(columns=[target])
y = df_numeric[target]

X = sm.add_constant(X)
X = X.replace([np.inf, -np.inf], np.nan).dropna()
y = y.loc[X.index]

model = sm.OLS(y, X).fit()
print(model.summary())

| Variable           | Coef          | p-value    | Interpretation                                                                                                                      |
| ------------------ | ------------- | ---------- | ----------------------------------------------------------------------------------------------------------------------------------- |
| `area_pop`         | **-1.6e-06**  | **0.001**  | Higher area population linked to **lower waste density** — possibly urban areas with better management.                             |
| `area_young_share` | **-0.145**    | **0.004**  | Higher share of young people associated with **lower waste density** — possibly younger communities generate less or manage better. |
| `young_area_pop`   | **+3.72e-06** | **0.004**  | But raw young population number **increases waste** — contradicts above, may point to a **nonlinear effect** or interaction.        |
| `density_gsm`      | **+1.15e-05** | **<0.001** | Higher GSM (mobile network) density linked to **more waste**, maybe a proxy for urban activity or infrastructure.                   |

## A few features

In [ ]:
# Pairplot of Selected Features (optional)
# selected = ['waste_per_image_per_km2', 'image_count', 'waste_pile_count', 'waste_density_per_km2', 'waste_per_image', 'grdi', 'rwi_weight', 'area_pop', 'area_young_share']
selected = ['waste_per_image_per_km2', 'grdi', 'density_gsm', 'area_pop']
sns.pairplot(df_numeric[selected])

In [ ]:
# Compare with a socioeconomic variable, like 'density_devices'
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

gdf_SE_summary.plot(column='waste_per_image_per_km2', cmap='OrRd', legend=True, ax=axes[0])
axes[0].set_title('Waste per Image per km²')
axes[0].axis('off')

gdf_SE_summary.plot(column='rwi_weight', cmap='Blues', legend=True, ax=axes[1])
axes[1].set_title('Density of Devices')
axes[1].axis('off')

plt.show()

## Clustering

- Cluster 0 → Moderate waste, medium socioeconomic indicators
- Cluster 1 → Low waste, low device/test density → possibly rural/less developed
- Cluster 2 → High waste, high density → possibly urban/affluent or high-activity areas

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df_numeric)

kmeans = KMeans(n_clusters=3, random_state=42)
gdf_SE_summary['cluster'] = kmeans.fit_predict(X_scaled)

# Visualize clusters
gdf_SE_summary.plot(column='cluster', categorical=True, legend=True, cmap='Set2', figsize=(10, 8))
plt.title('Clustered Regions Based on Waste and Socioeconomic Features')
plt.axis('off')
plt.show()

In [ ]:
cluster_summary = df_numeric.copy()
cluster_summary['cluster'] = gdf_SE_summary['cluster']
summary_by_cluster = cluster_summary.groupby('cluster').mean()
summary_by_cluster